In [2]:
import shap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

os.makedirs("../results/figures/shap", exist_ok=True)

C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Use mBERT (your best performing model)
MODEL_PATH = "../models/transformers/mBERT_finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)
model.eval()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(DEVICE)
print(f"✅ Model loaded on {DEVICE}")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 3143.86it/s]


✅ Model loaded on cuda


In [4]:
def predict_proba(texts):
    """
    Takes list of texts
    Returns numpy array of shape (n, 2)
    → column 0 = genuine probability
    → column 1 = fake probability
    """
    if isinstance(texts, str):
        texts = [texts]

    all_probs = []
    batch_size = 8

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors = "pt",
            truncation     = True,
            padding        = True,
            max_length     = 128
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(
            outputs.logits, dim=1
        ).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

# ── Test it works ─────────────────────────────────────────────
test_texts = [
    "Absolutely amazing product! Best purchase ever!!!",
    "khana ekdum mito thiyo, feri aaunchu pakkai"
]
probs = predict_proba(test_texts)
print("✅ Predict function works!")
for text, prob in zip(test_texts, probs):
    label = "FAKE" if prob[1] > 0.5 else "GENUINE"
    print(f"  {label} (genuine={prob[0]:.3f} fake={prob[1]:.3f})")
    print(f"  → {text[:60]}")

✅ Predict function works!
  FAKE (genuine=0.000 fake=1.000)
  → Absolutely amazing product! Best purchase ever!!!
  GENUINE (genuine=1.000 fake=0.000)
  → khana ekdum mito thiyo, feri aaunchu pakkai


In [6]:
# Load real test data
real_test = pd.read_csv("../data/processed/real_test.csv")

# Load raw synthetic for language style
df_syn_raw = pd.read_csv("../data/processed/processed_reviews.csv")
syn_test   = pd.read_csv("../data/processed/syn_test.csv")

syn_test_lang = syn_test.merge(
    df_syn_raw[["review_text", "language_style"]],
    on="review_text", how="left"
)

# ── Pick diverse samples for SHAP analysis ────────────────────
shap_samples = []

# 2 genuine per language
for lang in ["english", "romanized_nepali", "code_mixed"]:
    subset = syn_test_lang[
        (syn_test_lang["label"] == 0) &
        (syn_test_lang["language_style"] == lang)
    ].head(2)
    shap_samples.append(subset)

# 2 fake per language
for lang in ["english", "romanized_nepali", "code_mixed"]:
    subset = syn_test_lang[
        (syn_test_lang["label"] == 1) &
        (syn_test_lang["language_style"] == lang)
    ].head(2)
    shap_samples.append(subset)

shap_df = pd.concat(shap_samples, ignore_index=True)
shap_texts  = shap_df["review_text"].tolist()
shap_labels = shap_df["label"].tolist()

print(f"✅ SHAP samples prepared : {len(shap_texts)}")
print(f"   Genuine : {shap_labels.count(0)}")
print(f"   Fake    : {shap_labels.count(1)}")
print(f"\nSample reviews:")
for i, (t, l) in enumerate(zip(shap_texts[:3], shap_labels[:3])):
    print(f"  [{i}] {'FAKE' if l==1 else 'GENUINE'}: {t[:60]}...")

✅ SHAP samples prepared : 12
   Genuine : 6
   Fake    : 6

Sample reviews:
  [0] GENUINE: i call repair guy and then he come latee latee and check slo...
  [1] GENUINE: thik thik. not wow. food edible. place ok. nothing more...
  [2] GENUINE: Keema noodles 601 set paisa vasool thyo cha Pachi pheri try ...


In [7]:
# Use shap.Explainer with partition masker
# Best for transformer text models

masker = shap.maskers.Text(tokenizer)

explainer = shap.Explainer(
    predict_proba,
    masker,
    output_names=["genuine", "fake"]
)

print("✅ SHAP Explainer created!")
print("   Type : Partition explainer (best for transformers)")

✅ SHAP Explainer created!
   Type : Partition explainer (best for transformers)
